In [20]:
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import classification_report, accuracy_score

In [21]:
df = pd.read_csv('Dataset/archive/churn.csv')

In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  object 
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  object 
 5   Gender           10000 non-null  object 
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), object(3)
memory usage: 1.1+ MB


In [23]:
df.columns

Index(['RowNumber', 'CustomerId', 'Surname', 'CreditScore', 'Geography',
       'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
       'IsActiveMember', 'EstimatedSalary', 'Exited'],
      dtype='object')

In [24]:
df

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,15606229,Obijiaku,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,9997,15569892,Johnstone,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,9998,15584532,Liu,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,9999,15682355,Sabbatini,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [25]:
Label_encoder = LabelEncoder()

df['Gender'] = Label_encoder.fit_transform(df['Gender'])

In [26]:
df['Gender'].value_counts()

Gender
1    5457
0    4543
Name: count, dtype: int64

In [27]:
ohe = OneHotEncoder(sparse_output=False) 

geo_encoded = ohe.fit_transform(df[['Geography']])

In [28]:
geo_encoded

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]])

In [29]:
ohe.get_feature_names_out()

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [30]:
geo_df = pd.DataFrame(geo_encoded, columns=ohe.get_feature_names_out())

In [31]:
df = pd.concat([df, geo_df], axis=1)

In [32]:
df

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,1,15634602,Hargrave,619,France,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,2,15647311,Hill,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,3,15619304,Onio,502,France,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,4,15701354,Boni,699,France,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,5,15737888,Mitchell,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,15606229,Obijiaku,771,France,1,39,5,0.00,2,1,0,96270.64,0,1.0,0.0,0.0
9996,9997,15569892,Johnstone,516,France,1,35,10,57369.61,1,1,1,101699.77,0,1.0,0.0,0.0
9997,9998,15584532,Liu,709,France,0,36,7,0.00,1,0,1,42085.58,1,1.0,0.0,0.0
9998,9999,15682355,Sabbatini,772,Germany,1,42,3,75075.31,2,1,0,92888.52,1,0.0,1.0,0.0


In [33]:
df.columns

Index(['RowNumber', 'CustomerId', 'Surname', 'CreditScore', 'Geography',
       'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
       'IsActiveMember', 'EstimatedSalary', 'Exited', 'Geography_France',
       'Geography_Germany', 'Geography_Spain'],
      dtype='object')

In [34]:
X = df.drop(['RowNumber', 'CustomerId', 'Surname', 'Geography', 'Exited'], axis=1)
Y = df['Exited']

In [35]:
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.3, random_state=42)

In [36]:
scalar = StandardScaler()

X_train_scaled = scalar.fit_transform(X_train)
X_test_scaled = scalar.transform(X_test)

In [51]:
lr_param_grid = {
    'C': [0.01, 0.1, 1, 10],
    'penalty': ['l2', 'l1'],
    'solver': ['liblinear'],
    'class_weight': [None, 'balanced']
}

In [54]:
lr_grid = GridSearchCV(
    estimator=LogisticRegression(max_iter=1000),
    param_grid=lr_param_grid,
    scoring='recall',
)

In [55]:
lr_grid.fit(X_train_scaled, y_train)

GridSearchCV(estimator=LogisticRegression(max_iter=1000),
             param_grid={'C': [0.01, 0.1, 1, 10],
                         'class_weight': [None, 'balanced'],
                         'penalty': ['l2', 'l1'], 'solver': ['liblinear']},
             scoring='recall')

In [47]:
d_tree = DecisionTreeClassifier(random_state=42)

dt_param_grid = {
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'criterion': ['gini', 'entropy'],
    'class_weight': [None, 'balanced']
}

In [48]:
dt_grid = GridSearchCV(
    d_tree,
    dt_param_grid,
    scoring='recall',
)

dt_grid.fit(X_train, y_train)

GridSearchCV(estimator=DecisionTreeClassifier(random_state=42),
             param_grid={'class_weight': [None, 'balanced'],
                         'criterion': ['gini', 'entropy'],
                         'max_depth': [None, 5, 10, 20],
                         'min_samples_leaf': [1, 2, 4],
                         'min_samples_split': [2, 5, 10]},
             scoring='recall')

In [60]:
r_forest = RandomForestClassifier(random_state=42)

rf_param_grid = {
    'n_estimators': [10, 20, 100],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2'],
    'class_weight': [None, 'balanced']
}

In [61]:
rf_grid = GridSearchCV(
    r_forest,
    rf_param_grid,
    scoring='recall',
)

rf_grid.fit(X_train, y_train)

GridSearchCV(estimator=RandomForestClassifier(random_state=42),
             param_grid={'class_weight': [None, 'balanced'],
                         'max_depth': [None, 5, 10],
                         'max_features': ['sqrt', 'log2'],
                         'min_samples_leaf': [1, 2],
                         'min_samples_split': [2, 5],
                         'n_estimators': [10, 20, 100]},
             scoring='recall')

In [62]:
def evaluate_model(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test)

    print(f"\n{model_name}")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred))

In [63]:
evaluate_model(lr_grid.best_estimator_, X_test, y_test, "Logistic Regression")
evaluate_model(dt_grid.best_estimator_, X_test, y_test, "Decision Tree")
evaluate_model(rf_grid.best_estimator_, X_test, y_test, "Random Forest")



Logistic Regression
Accuracy: 0.19466666666666665
              precision    recall  f1-score   support

           0       0.00      0.00      0.00      2416
           1       0.19      1.00      0.33       584

    accuracy                           0.19      3000
   macro avg       0.10      0.50      0.16      3000
weighted avg       0.04      0.19      0.06      3000


Decision Tree
Accuracy: 0.7213333333333334
              precision    recall  f1-score   support

           0       0.93      0.70      0.80      2416
           1       0.39      0.79      0.53       584

    accuracy                           0.72      3000
   macro avg       0.66      0.75      0.66      3000
weighted avg       0.83      0.72      0.75      3000


Random Forest
Accuracy: 0.7793333333333333
              precision    recall  f1-score   support

           0       0.92      0.79      0.85      2416
           1       0.46      0.72      0.56       584

    accuracy                           0.78

c:\Users\ABC\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
c:\Users\ABC\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\ABC\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\ABC\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py

In [64]:
churn_random_forest = rf_grid.best_estimator_

In [65]:
rf_grid.best_params_

{'class_weight': 'balanced',
 'max_depth': 5,
 'max_features': 'sqrt',
 'min_samples_leaf': 2,
 'min_samples_split': 5,
 'n_estimators': 10}

In [70]:
X_test

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
6252,596,1,32,3,96709.07,2,0,0,41788.37,0.0,1.0,0.0
4684,623,1,43,1,0.00,2,1,1,146379.30,1.0,0.0,0.0
1731,601,0,44,4,0.00,2,1,0,58561.31,0.0,0.0,1.0
4742,506,1,59,8,119152.10,2,1,1,170679.74,0.0,1.0,0.0
4521,560,0,27,7,124995.98,1,1,1,114669.79,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...
8014,731,0,39,2,126816.18,1,1,1,74850.93,0.0,0.0,1.0
1074,535,0,31,7,111855.04,2,1,1,36278.89,1.0,0.0,0.0
3063,609,1,32,7,71872.19,1,1,1,151924.90,1.0,0.0,0.0
6487,580,1,31,2,0.00,2,0,1,64014.24,1.0,0.0,0.0


In [108]:
y_test.loc[0]

1

In [109]:
X_test.loc[0]

CreditScore             619.00
Gender                    0.00
Age                      42.00
Tenure                    2.00
Balance                   0.00
NumOfProducts             1.00
HasCrCard                 1.00
IsActiveMember            1.00
EstimatedSalary      101348.88
Geography_France          1.00
Geography_Germany         0.00
Geography_Spain           0.00
Name: 0, dtype: float64

Test data:

Credit Score: 502  
Gender: Female  
Age: 42	  
Tenure: 8  	
Balance: 159660.80  	
Num_preducts: 3	  
Has_creditcard: 1  	
IsActive: 0	  
EstimatedSalary: 113931.57  	
Geography: France  

----------------  
Exited  ---> 1 (churn)  
----------------  

In [110]:
credit_score = int(input('Enter CreditScore: '))  # 596

# I have label encoded Gender
gender = input("Enter Gender")  # Male/Female
gender = Label_encoder.transform([gender])[0]

age = int(input('Enter Age')) # 32

Tenure = float(input("Enter Tenure"))  # 3

Balance = float(input("Enter Balance:"))  # 96709.07

Num_products = int(input("Enter Num products: "))  # 2

has_credit_card = int(input("Has credit card (Enter 0/1): ")) # 0

is_active = int(input("Is active (0/1): "))  # 0

Estimated_salary = float(input("Enter Estimated salary: ")) # 41788.37

geography = input("Enter Geography [France, Germany, Spain]")  # Germany
geography = ohe.transform([[geography]])[0]


churn = churn_random_forest.predict([[credit_score, gender, age, Tenure, Balance, Num_products, has_credit_card, is_active, Estimated_salary, geography[0], geography[1], geography[2]]])

print("churn: ", churn)

churn:  [1]


c:\Users\ABC\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
c:\Users\ABC\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [66]:
import joblib

In [67]:
joblib.dump(churn_random_forest, 'churn_pred_rf_model.pkl')

['churn_pred_rf_model.pkl']

In [111]:
joblib.dump(Label_encoder, 'gender_label_encoder.pkl')

['gender_label_encoder.pkl']

In [112]:
joblib.dump(ohe, 'geography_ohe.pkl')

['geography_ohe.pkl']